In [3]:
with open("input.txt", "r", encoding='utf-8') as f:
    text = f.read()

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {s:i for i,s in enumerate(chars)}

itos = {i:s for i,s in enumerate(chars)}

print(stoi)
print(itos)

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars)}")

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i',

In [44]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [45]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd):
        super().__init__()
        self.token_embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
        self.position_embedding_table = nn.Embedding(num_embeddings=block_size, embedding_dim=n_embd)
    def forward(self, idx):
        B, T = idx.shape
        tok_embd = self.token_embedding_table(idx)
        pos_embd = self.position_embedding_table(torch.arange(T))
        sum = tok_embd + pos_embd
        return sum

In [46]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        # K, Q, V Vectors
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query = nn.Linear(n_embd, head_size, bias = False)
        self.value = nn.Linear(n_embd, head_size, bias = False)

        # Causal Mask
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # attention scores
        affinities = q @ k.transpose(-2, -1)

        #scaling 
        affinities = affinities / (k.shape[-1] ** 0.5)

        # masking
        logits = affinities.masked_fill_(self.tril[:T, :T] == 0, float('-inf'))

        # activation func
        act = F.softmax(logits, dim=-1)
    
        out = act @ v

        return out

        

In [47]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.linear = nn.Linear(num_heads * head_size, num_heads * head_size)

    def forward(self, x):
        # Calling forward pass for every head in list of heads
        out_list = [h(x) for h in self.heads]

        # concatenating the list of heads together
        cat = torch.cat(out_list, dim=-1)
        
        out = self.linear(cat)

        return out

In [48]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        # Creating sequential network
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd)
        )
    def forward(self, x):
        # forward pass
        return self.net(x)

In [49]:
class Block(nn.Module):
    def __init__(self, n_embd, num_heads):
        super().__init__()
        
        head_size = n_embd // num_heads

        self.multi = MultiHeadAttention(num_heads, head_size)
        self.feed = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        
    def forward(self, x):
        # attention
        out1 = self.multi(self.ln1(x))
        x = x + out1

        # feed forward prediction
        out2 = self.feed(self.ln2(x))
        x = x + out2

        return x


In [50]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd, n_heads, n_layers):
        super().__init__()

        self.block_size = block_size

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # multi block for more context
        block_list = [Block(n_embd, n_heads) for _ in range(n_layers)]
        self.blocks = nn.Sequential(*block_list)
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))
        x = tok_emb + pos_emb

        # forward pass
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        return logits
        
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]

            # forward
            logits = self(idx_cond)
            
            logits = logits[:, -1, :] 

            # softmax for probs
            probs = F.softmax(logits, dim=-1)
            
            # Sample using torch.multinomial
            idx_next = torch.multinomial(probs, num_samples=1)
            
            idx = torch.cat((idx, idx_next), dim=1)
            
        return idx

In [51]:
# hyper params
vocab_size = 65
block_size = 8
n_embd = 32
n_heads = 4
n_layers = 3

batch_size = 32
epochs = 5000

model = GPTLanguageModel(vocab_size, block_size, n_embd, n_heads, n_layers)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print(f"Model has {sum(p.numel() for p in model.parameters()) / 1e6:.3f} Million parameters")

Model has 0.042 Million parameters


In [52]:
# Converting Shakespeare text into a tensor of integers
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

# 90% train, 10% validation split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split, batch_size, block_size):
    data = train_data if split == 'train' else val_data
    
    # Generate random starting indexes
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    # Stack the chunks into a batch
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [53]:
# training
for k in range(epochs):
    # takes batch
    xb, yb = get_batch('train', batch_size, block_size)

    # forward
    logits = model(xb)
    
    # loss
    B, T, C = logits.shape
    logits_reshaped = logits.view(B * T, C)
    targets_reshaped = yb.view(B * T)
    
    loss = F.cross_entropy(logits_reshaped, targets_reshaped)
    
    # Backpropagate and Update
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    # Print progress every 1000 steps
    if k % 1000 == 0:
        print(f"Epoch {k} | Loss: {loss.item():.4f}")
        
print(f"Training complete, final loss: {loss.item()}")

Epoch 0 | Loss: 4.2378
Epoch 1000 | Loss: 2.3960
Epoch 2000 | Loss: 2.2001
Epoch 3000 | Loss: 2.1477
Epoch 4000 | Loss: 1.8993
Training complete


In [55]:
# sampling generation
context = torch.zeros((1, 1), dtype=torch.long)

generated_indices = model.generate(context, max_new_tokens=500)

generated_text = "".join([itos[i] for i in generated_indices.tolist()[0]])

print("--- AI SHAKESPEARE ---")
print(generated_text)

--- AI SHAKESPEARE ---

If sligh how;
I avil.

FRCICHARD Yogursent
Hervany thouu!
How, libste,
My deveid to conges I what breather'T pery that shall have she a gentreens fay mutanded 't bless father' suit a bitind's are his donty be sose! If gaded or hant that tyou be;
To deid bgo RoRmek should a past you beer frace theirst fark frofe shally turn letble tits the sastesserd hirs
Dufar memight,
To? be shall
Showhich sme toll is not world camate-hene king not
By vince,
Yeast such a maning. O held do wife eyesh?
Helge as C
